# Building Custom Tools (LangChain @tool)
## Problem Statement
Two tools - diff_parser and dependency_finder - wired into a ReAct agent for PR impact analysis.

## My Approach
Two tools in sequence: diff_parser pulls changed function names from the raw diff, then dependency_finder scans the codebase index for callers of those functions. The agent knows the order because the docstrings say so explicitly - "call this first", "call this after". No orchestration code needed. The codebase index is a list of document chunks with metadata, exactly as ChromaDB returns them. The tool scans those chunks for function call patterns and returns file paths with line numbers.

## What I Learned
The docstring is the tool's API contract with the LLM - vague docstrings cause wrong tool selection or hallucinated arguments.

## Where I Got Stuck
The .lower() on the scanned line before the regex search - I lowercased the target string but kept the pattern case-sensitive. For validate_token it worked by coincidence (already lowercase). For validateToken it would silently return nothing. 

The fix: lowercase both the search line and the function name in the pattern. Also the empty tool result - dependency_finder returning [] caused Groq to 400 because role:tool messages must have non-empty content. The real fix is ensuring the mock data actually contains callable text, not a dict repr.

## What I'd Do Differently
Test each tool in isolation with .invoke() before wiring the agent - a direct call would have surfaced the empty return in 10 seconds. Also start with realistic mock data from day one: document chunks with text + metadata fields, not a key-value dict. That forces the tool logic to match production shape and avoids having to rewrite it when ChromaDB comes in picture.

## My Solution (Naive)
*First attempt - written before reviewing feedback*

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

from langchain_core.tools import tool
from langchain.agents import create_agent
import re
from langgraph.errors import GraphRecursionError

d:\Python\genai-engineering-portfolio\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Python\genai-engineering-portfolio\venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [27]:
SAMPLE_DIFF = """diff --git a/auth/token_validator.py b/auth/token_validator.py
--- a/auth/token_validator.py
+++ b/auth/token_validator.py
@@ -12,7 +12,9 @@
-def validate_token(token: str) -> bool:
+def validate_token(token: str, strict: bool = False) -> bool:
     \"\"\"Validates JWT token against secret key.\"\"\"
"""

MOCK_REGISTRY = [
    {
        "text": "result = validate_token(request.headers.get('Authorization'))",
        "metadata": {"source": "auth/middleware.py", "line": 34}
    },
    {
        "text": "if not validate_token(token):\n    raise HTTPException(status_code=401)",
        "metadata": {"source": "api/routes/login.py", "line": 88}
    },
    {
        "text": "assert validate_token('test-jwt-xyz') == True",
        "metadata": {"source": "tests/test_auth.py", "line": 12}
    },
    {
        "text": "new_token = refresh_token(user_id=session.user_id)",
        "metadata": {"source": "api/routes/session.py", "line": 55}
    },
]

In [34]:
# Define @tool functions
@tool
def diff_parser (diff_txt:str) -> list[str]:
    """
    This function is designed to get all changes in git string .
    Input is raw git unified diff ( output of `git diff` ) and output is list of modified function/method name.
    in case of no function/method name , found return empty list .
    """
    pattern = r'^[+-][\t ]*def\s+(\w+)\s*\('
    matches=re.findall(pattern , diff_txt , re.MULTILINE)
    return list(set(matches))

@tool
def dependency_finder(function_name : str ) -> list[str] :
    """
    This function finds all code snippets in MOCK_REGISTRY on basis of name given in function_name given .
    This function is called after diff_parser to identify which parts of code base depends on changed function i.e blast radius of change .
    Input is function name to be found in codebase 
    Output is list of code snippets which contain calls to function.
    """
    # \b ensures word boundary so i do not get reauth() if i am searching for auth()
    fn= function_name.strip().lower()
    call_pattern = re.compile(fr'\b{re.escape(fn)}\s*\(') 
    calling_lines=[]
    for doc in MOCK_REGISTRY:
        doc_lower= doc["text"].lower()
        if call_pattern.search(doc_lower):
            calling_lines.append(f"{doc['metadata']['source']}:{doc['metadata']['line']}")
    return calling_lines
   


In [35]:
print("Tool name:", diff_parser.name)           
print("Description:", diff_parser.description)  
print("Schema:", diff_parser.args_schema.model_json_schema())
print()
print("Tool name:", dependency_finder.name)           
print("Description:", dependency_finder.description)  
print("Schema:", dependency_finder.args_schema.model_json_schema())

Tool name: diff_parser
Description: This function is designed to get all changes in git string .
Input is raw git unified diff ( output of `git diff` ) and output is list of modified function/method name.
in case of no function/method name , found return empty list .
Schema: {'description': 'This function is designed to get all changes in git string .\nInput is raw git unified diff ( output of `git diff` ) and output is list of modified function/method name.\nin case of no function/method name , found return empty list .', 'properties': {'diff_txt': {'title': 'Diff Txt', 'type': 'string'}}, 'required': ['diff_txt'], 'title': 'diff_parser', 'type': 'object'}

Tool name: dependency_finder
Description: This function finds all code snippets in MOCK_REGISTRY on basis of name given in function_name given .
This function is called after diff_parser to identify which parts of code base depends on changed function i.e blast radius of change .
Input is function name to be found in codebase 
Outp

In [36]:
def print_trace(messages: list) -> None:
    """Print a clean ReAct trace from the messages list."""
    tool_call_count = 0
    for i, msg in enumerate(messages):
        role = msg.__class__.__name__
        if role == "HumanMessage":
            continue
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                tool_call_count += 1
                print(f"  [Step {i}] TOOL CALL: {tc['name']}({tc['args']})")
        elif hasattr(msg, "content") and msg.content:
            preview = str(msg.content)[:150]
            print(f"  [Step {i}] {role}: {preview}")
    print(f"  Total tool calls: {tool_call_count}")

In [37]:
# Wire agent + invoke
query = f"""
I have a git diff and some source code. 
First use diff_parser to find what function changed in this diff:
{SAMPLE_DIFF}

Then use dependency_finder to find which lines in this source code call that function.

Summarise the blast radius of this change.
"""

SYSTEM_PROMPT = (
    "You are DevContext AI - a developer assistant for codebase onboarding and PR impact analysis. "
    "Use diff_parser for getting all changes from git string. "
    "Use dependency_finder after diff_parser to find all code snippets in source code on basis of name of function name obtained"
    "When a query needs both, call both tools before answering. "
    "If a tool returns empty list, report it clearly - do not guess or invent information."
)
agent = create_agent(
            llm 
            ,tools=[diff_parser , dependency_finder] 
            ,system_prompt=SYSTEM_PROMPT
)


In [38]:
try : 
    print("Running ReAct agent over all queries...\n")
    print("=" * 70)
    
    result=agent.invoke(
        {"messages" : [{"role":"user" , "content" : query}] } 
        ,config={"recursion_limit": 10}
    )
    final_answer = result["messages"][-1].content
    print(f"Answer: {final_answer}")
    print("\nTrace:")
    print_trace(result["messages"])
except GraphRecursionError:
    print("ERROR: Agent hit recursion limit - tool loop detected. Check tool docstrings.")
except Exception as e:
    print(f"ERROR: {e}")
print("-" * 70)

Running ReAct agent over all queries...

Answer: The function `validate_token` has been modified in the given git diff. The blast radius of this change includes the following lines of code that call the `validate_token` function: `auth/middleware.py:34`, `api/routes/login.py:88`, and `tests/test_auth.py:12`. These lines may be affected by the change to the `validate_token` function.

Trace:
  [Step 1] TOOL CALL: diff_parser({'diff_txt': 'diff --git a/auth/token_validator.py b/auth/token_validator.py\n--- a/auth/token_validator.py\n+++ b/auth/token_validator.py\n@@ -12,7 +12,9 @@\n-def validate_token(token: str) -> bool:\n+def validate_token(token: str, strict: bool = False) -> bool:\n     """Validates JWT token against secret key."""\n'})
  [Step 1] TOOL CALL: dependency_finder({'function_name': 'validate_token'})
  [Step 2] ToolMessage: ["validate_token"]
  [Step 3] ToolMessage: ["auth/middleware.py:34", "api/routes/login.py:88", "tests/test_auth.py:12"]
  [Step 4] AIMessage: The func